In [5]:
%pip install pandas numpy matplotlib pyarrow

Note: you may need to restart the kernel to use updated packages.


In [14]:
import sqlite3
import pandas as pd
import numpy as np
import os
import logging
from pathlib import Path
from datetime import datetime

# ── Logging setup ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('etl')

# ── File paths ─────────────────────────────────────────────────────────────────
# Changed DATA_DIR to /content to match your file structure
DATA_DIR = Path('.')
DB_PATH       = DATA_DIR / 'PerfectPetClub.db'
CRM_PATH      = DATA_DIR / 'PerfectPetClub_CRM_report.csv'
LOG_PATH      = DATA_DIR / 'PerfectPetClub_server_log.csv'
SURVEY_PATH   = DATA_DIR / 'PerfectPetClub_survey (2).csv'
OUTPUT_PARQUET = DATA_DIR / 'churn_dataset.parquet'

# Reference date for recency calculations (last day with sales in the dataset)
REFERENCE_DATE = pd.Timestamp('2026-11-10')

logger.info('Configuration loaded. Reference date: %s', REFERENCE_DATE.date())

17:47:32 [INFO] Configuration loaded. Reference date: 2026-11-10


In [16]:
def extract_db(db_path: Path) -> dict:
    """
    Extract all relevant tables from the PerfectPetClub SQLite database.

    Parameters
    ----------
    db_path : Path
        File-system path to the .db file.

    Returns
    -------
    dict
        Dictionary of {table_name: DataFrame} for each extracted table.
    """
    logger.info('EXTRACT: connecting to SQLite database at %s', db_path)
    conn = sqlite3.connect(db_path)

    tables = {
        'customer':          'SELECT * FROM customer',
        'subscription':      'SELECT * FROM subscription',
        'subscription_type': 'SELECT * FROM subscription_type',
        'sale':              'SELECT * FROM sale',
        'payment':           'SELECT * FROM payment',
        'sale_product':      'SELECT * FROM sale_product',
        'product':           'SELECT * FROM product',
        'product_category':  'SELECT * FROM product_category',
        'product_review':    'SELECT * FROM product_review',
    }

    raw = {}
    for name, query in tables.items():
        raw[name] = pd.read_sql_query(query, conn)
        logger.info('  Extracted table %-20s shape: %s', name, raw[name].shape)

    conn.close()
    logger.info('EXTRACT DB: complete – %d tables loaded', len(raw))
    return raw


def extract_crm(crm_path: Path) -> pd.DataFrame:
    """
    Extract the CRM customer service contact report from CSV.

    Parameters
    ----------
    crm_path : Path
        Path to PerfectPetClub_CRM_report.csv

    Returns
    -------
    pd.DataFrame
        Raw CRM DataFrame.
    """
    logger.info('EXTRACT: reading CRM report from %s', crm_path)
    df = pd.read_csv(crm_path)
    logger.info('EXTRACT CRM: shape %s', df.shape)
    return df


def extract_server_log(log_path: Path) -> pd.DataFrame:
    """
    Extract the web application server login log from CSV.

    Parameters
    ----------
    log_path : Path
        Path to PerfectPetClub_server_log.csv

    Returns
    -------
    pd.DataFrame
        Raw server log DataFrame.
    """
    logger.info('EXTRACT: reading server log from %s', log_path)
    df = pd.read_csv(log_path)
    logger.info('EXTRACT SERVER LOG: shape %s', df.shape)
    return df


def extract_survey(survey_path: Path) -> pd.DataFrame:
    """
    Extract customer survey responses from CSV.
    If the file is not found, returns an empty DataFrame with expected columns
    so that downstream transformations remain robust.

    Parameters
    ----------
    survey_path : Path
        Path to PerfectPetClub_survey.csv

    Returns
    -------
    pd.DataFrame
        Raw survey DataFrame, or empty DataFrame if file unavailable.
    """
    if not survey_path.exists():
        logger.warning('Survey file not found at %s – returning empty DataFrame', survey_path)
        return pd.DataFrame(columns=['email', 'reported_income',
                                     'nps_recommend', 'satisfaction_score',
                                     'online_comparison_freq'])
    logger.info('EXTRACT: reading survey from %s', survey_path)
    df = pd.read_csv(survey_path)
    logger.info('EXTRACT SURVEY: shape %s', df.shape)
    return df

In [17]:
# ── Run all extractions ────────────────────────────────────────────────────────
db_tables  = extract_db(DB_PATH)
raw_crm    = extract_crm(CRM_PATH)
raw_log    = extract_server_log(LOG_PATH)
raw_survey = extract_survey(SURVEY_PATH)

# Quick sanity check – print head of each raw source
print('\n=== Customer (DB) ===')
display(db_tables['customer'].head(3))

print('\n=== Subscription (DB) ===')
display(db_tables['subscription'].head(3))

print('\n=== CRM ===')
display(raw_crm.head(3))

print('\n=== Server Log ===')
display(raw_log.head(3))

17:47:49 [INFO] EXTRACT: connecting to SQLite database at PerfectPetClub.db
17:47:49 [INFO]   Extracted table customer             shape: (221, 9)
17:47:49 [INFO]   Extracted table subscription         shape: (221, 5)
17:47:49 [INFO]   Extracted table subscription_type    shape: (3, 3)
17:47:49 [INFO]   Extracted table sale                 shape: (23188, 6)
17:47:49 [INFO]   Extracted table payment              shape: (23188, 4)
17:47:49 [INFO]   Extracted table sale_product         shape: (59759, 4)
17:47:49 [INFO]   Extracted table product              shape: (194, 4)
17:47:49 [INFO]   Extracted table product_category     shape: (26, 2)
17:47:49 [INFO]   Extracted table product_review       shape: (10990, 5)
17:47:49 [INFO] EXTRACT DB: complete – 9 tables loaded
17:47:49 [INFO] EXTRACT: reading CRM report from PerfectPetClub_CRM_report.csv
17:47:49 [INFO] EXTRACT CRM: shape (7585, 4)
17:47:49 [INFO] EXTRACT: reading server log from PerfectPetClub_server_log.csv
17:47:49 [INFO] EXTRAC


=== Customer (DB) ===


,id,name,email,password_hash,address_line_1,address_line_2,county,postcode,subscription_id
0,1,Anahit Sargsyan,Sargsyan701@somemail.co.uk,6741fa5591efb6a5310f241a22ca305eb0449a4bb3dd20...,76 High Street,Maidenhead,Berkshire,SL6 1CD,1
1,2,Leyla Mammadova,Mammadova282@example.com,3931efc8f036b658429849e8b951f462f750d1c6b15536...,18 High Street,Chipping Norton,Oxfordshire,OX7 5GH,2
2,3,Sana Shah,Sana858@notrealemail.com,db0af1e14f9d10b440a960e2de9ada9ad83d92449189f1...,77 High Street,Kidderminster,Worcestershire,DY10 2JK,3



=== Subscription (DB) ===


,id,start_date,end_date,auto_renew,subscription_type_id
0,1,2026-01-01 00:00:00.000,2026-01-31 00:00:00.000,0,3
1,2,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,3
2,3,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,1



=== CRM ===


,customer_id,recieved_datatime,platform,type
0,55,2026-01-01 00:42:47.000,['e-mail'],complaint
1,49,2026-01-01 01:25:34.000,['live-chat'],query
2,44,2026-01-01 03:33:55.001,['telephone'],query



=== Server Log ===


,datetime,session_id,ip,login,user_agent
0,2026-01-01 02:08:21.000,session_cookie::63fec84dedaa4f2cbaa10b413b5a7d...,239.66.231.182,Sipho928@somemail.co.uk,"""Mozilla/5.0 (Desktop/Laptop; Cipher CoreBook ..."
1,2026-01-01 02:51:08.001,session_cookie::64bcd32ed1494bd1af6b7a233cd638...,200.200.140.49,Sienna.Martin923@notrealemail.com,"""Mozilla/5.0 (Mobile; Pulse PulsePhone) AppleW..."
2,2026-01-01 04:16:42.001,session_cookie::f848746c9c96422b9eb050eccacfeb...,38.219.218.103,v134@somemail.co.uk,"""Mozilla/5.0 (Mobile; Titan Pocket Go) AppleWe..."
